# Tsurugi NNUE training (Kaggle GPU)

Trains Tsurugi's 768 -> 2x256 -> 1 network on quiet positions from the CC0 Lichess evaluation database
(Stockfish evaluations), using the same scripts as the local pipeline (`tools/nnue/`).

Settings needed (right panel): **Accelerator: GPU T4 x2 or P100**, **Internet: on**. Then *Run All*.
Expected time: ~3-4 h data preparation (streams 22 GB from database.lichess.org, never stored) + ~1 h training.
The trained network appears under **Output** as `tsurugi-v2.nnue` (+ `tsurugi-v2.json` training log).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2
!rm -rf tsurugi && git clone --depth 1 https://github.com/Abhayindia/tsurugi.git
!pip install -q chess zstandard

In [ ]:
POSITIONS = 60_000_000  # quiet positions to extract; ~67 bytes each on disk
!python tsurugi/scripts/nnue/prepare_data.py --src https://database.lichess.org/lichess_db_eval.jsonl.zst --out /tmp/nnue --positions {POSITIONS} --workers 4

In [ ]:
!python tsurugi/scripts/nnue/train.py --data /tmp/nnue --out /kaggle/working/tsurugi-v2.nnue --epochs 20 --batch 16384 --threads 4

In [ ]:
# Build the engine from source and check the network gives exactly the engine's evaluation.
!cd tsurugi && cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j4 --target tsurugi > /dev/null && echo built
!python tsurugi/scripts/nnue/check_net.py tsurugi/build/tsurugi /kaggle/working/tsurugi-v2.nnue
!cp /tmp/nnue/meta.json /kaggle/working/tsurugi-v2-data.json && ls -la /kaggle/working

Download `tsurugi-v2.nnue`, `tsurugi-v2.json` and `tsurugi-v2-data.json` from **Output** and save them in
`D:\tsurugi\nets\`. The local check-in will verify it again and queue it for an SPRT test against
the current network.